# 03 – Figures

Publication-quality figures comparing generated scenarios against
historical data and visualising attribution results.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

sns.set_theme(style='whitegrid', palette='muted')

data_cfg   = OmegaConf.load('../configs/data.yaml')
sample_cfg = OmegaConf.load('../configs/sample.yaml')

processed_dir = Path('../') / data_cfg.processed_dir
samples_dir   = Path('../') / sample_cfg.samples_dir
figures_dir   = Path('../outputs/figures')
tables_dir    = Path('../outputs/tables')
figures_dir.mkdir(parents=True, exist_ok=True)

## 1. Marginal return distributions – real vs. generated

In [ ]:
hist_df = pd.read_csv(processed_dir / 'returns_normalised.csv', index_col=0, parse_dates=True)
n = len(hist_df)
test_start = int(n * (data_cfg.split.train + data_cfg.split.val))
hist_test = hist_df.values[test_start:]

raw = np.load(samples_dir / sample_cfg.samples_file)
gen = raw['scenarios'][:, :, -1]  # last-day return

asset_names = list(data_cfg.tickers)

fig, axes = plt.subplots(1, len(asset_names), figsize=(4 * len(asset_names), 4))
for ax, col, i in zip(axes, asset_names, range(len(asset_names))):
    ax.hist(hist_test[:, i], bins=60, density=True, alpha=0.55, label='Historical', color='steelblue')
    ax.hist(gen[:, i], bins=80, density=True, alpha=0.5, label='Generated', color='tomato')
    ax.set_title(col)
    ax.set_xlabel('Normalised return')
axes[0].legend()
fig.suptitle('Marginal distributions: real vs. generated', y=1.02)
plt.tight_layout()
plt.savefig(figures_dir / 'marginal_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. VaR / ES comparison bar chart

In [ ]:
risk_table = pd.read_csv(tables_dir / 'var_es_comparison.csv')

x = np.arange(len(risk_table))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ['VaR', 'ES']):
    ax.bar(x - width/2, risk_table[f'hist_{metric}'], width, label='Historical', color='steelblue', alpha=0.8)
    ax.bar(x + width/2, risk_table[f'gen_{metric}'], width, label='Generated', color='tomato', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([f'α={a:.0%}' for a in risk_table['alpha']])
    ax.set_title(f'Portfolio {metric}')
    ax.legend()
    ax.set_ylabel(metric)

plt.tight_layout()
plt.savefig(figures_dir / 'var_es_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Attribution bar chart

In [ ]:
attr_table = pd.read_csv(tables_dir / 'attribution.csv', index_col=0)

fig, axes = plt.subplots(1, len(attr_table.columns), figsize=(5 * len(attr_table.columns), 4))
if len(attr_table.columns) == 1:
    axes = [axes]
for ax, col in zip(axes, attr_table.columns):
    attr_table[col].sort_values().plot.barh(ax=ax, color='steelblue')
    ax.set_title(col.replace('_', ' '))
    ax.set_xlabel('Attribution value')

plt.tight_layout()
plt.savefig(figures_dir / 'attribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Spearman correlation heatmaps

In [ ]:
from scipy import stats

hist_corr, _ = stats.spearmanr(hist_test)
gen_corr, _  = stats.spearmanr(gen)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
kw = dict(annot=True, fmt='.2f', cmap='RdYlGn', vmin=-1, vmax=1,
          xticklabels=asset_names, yticklabels=asset_names)
sns.heatmap(hist_corr, ax=axes[0], **kw)
axes[0].set_title('Historical Spearman ρ')
sns.heatmap(gen_corr, ax=axes[1], **kw)
axes[1].set_title('Generated Spearman ρ')
sns.heatmap(np.abs(hist_corr - gen_corr), ax=axes[2],
            annot=True, fmt='.2f', cmap='Reds', vmin=0,
            xticklabels=asset_names, yticklabels=asset_names)
axes[2].set_title('|Difference|')

plt.tight_layout()
plt.savefig(figures_dir / 'spearman_corr.png', dpi=150, bbox_inches='tight')
plt.show()